# Cluster-specific TFT streamflow forecasting — corrected Kaggle notebook

This notebook fixes the NaN-producing attention mask and adds safeguards against silent data, scaling, checkpoint, and evaluation failures.

### Kaggle setup
1. Upload/add the dataset containing `prepared_data` to the notebook.
2. Enable **GPU** under **Settings → Accelerator**.
3. In the configuration cell, either leave `DATA_ROOT = None` for automatic discovery or enter the full path, for example:
   `/kaggle/input/my-hydrology-data/prepared_data`
4. Run all cells from the top.
5. Start with `MAX_EPOCHS = 5` and one cluster. After the diagnostic run succeeds, increase the epochs and train all clusters.

Old checkpoints created by the defective mask must not be reused. This notebook uses a new output directory by default.

In [1]:
# ── Cell 1: Create Cluster_0 (global) in writable /kaggle/working/ ───────────
# Kaggle /kaggle/input/ is READ-ONLY — all new folders must go in /kaggle/working/

from pathlib import Path
import shutil, os

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tripathivaib/streamflow-forecating-tft-4th-obj/prepared_data"
)
WORK_DATA_ROOT  = Path("/kaggle/working/prepared_data_global")   # writable copy

# ── (Re)build writable working copy ───────────────────────────────────────────
if WORK_DATA_ROOT.exists():
    shutil.rmtree(WORK_DATA_ROOT)
WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 1. Copy root support files (feature_config.json, Station_Quality_Classified.csv, etc.)
for src_file in INPUT_DATA_ROOT.iterdir():
    if src_file.is_file():
        shutil.copy2(src_file, WORK_DATA_ROOT / src_file.name)
        print(f"  Copied root file: {src_file.name}")

feature_config = WORK_DATA_ROOT / "feature_config.json"
if not feature_config.exists():
    raise FileNotFoundError("feature_config.json not found — check INPUT_DATA_ROOT path.")

# 2. Copy Cluster_1 to Cluster_4 into working directory
for cl in [1, 2, 3, 4]:
    src_cl = INPUT_DATA_ROOT / f"Cluster_{cl}"
    dst_cl = WORK_DATA_ROOT  / f"Cluster_{cl}"
    if not src_cl.exists():
        raise FileNotFoundError(f"Source cluster folder not found: {src_cl}")
    if dst_cl.exists():
        shutil.rmtree(dst_cl)
    shutil.copytree(src_cl, dst_cl)
    n_files = sum(1 for _ in dst_cl.rglob("*") if _.is_file())
    print(f"  Cluster_{cl} copied: {n_files} files → {dst_cl}")

# 3. Build Cluster_0 (global = all basins merged) in writable location
GLOBAL_DIR    = WORK_DATA_ROOT / "Cluster_0"
GLOBAL_STREAM = GLOBAL_DIR / "streamflow"
GLOBAL_FORCE  = GLOBAL_DIR / "forcing"
for d in [GLOBAL_STREAM, GLOBAL_FORCE]:
    d.mkdir(parents=True, exist_ok=True)

# Copy streamflow + forcing from all 4 clusters into Cluster_0
copied_s, copied_f, skipped = 0, 0, []
seen_stations = set()

for cl in [1, 2, 3, 4]:
    src_s = WORK_DATA_ROOT / f"Cluster_{cl}" / "streamflow"
    src_f = WORK_DATA_ROOT / f"Cluster_{cl}" / "forcing"

    # Read station list for this cluster
    slist_path = WORK_DATA_ROOT / f"Cluster_{cl}" / "station_list.txt"
    if slist_path.exists():
        stations = [l.strip() for l in open(slist_path).readlines() if l.strip()]
    else:
        stations = [p.stem for p in src_s.glob("*.csv")]

    for sid in stations:
        if sid in seen_stations:
            skipped.append(sid)
            continue
        seen_stations.add(sid)

        sf = src_s / f"{sid}.csv"
        ff = src_f / f"{sid}.csv"
        if sf.exists():
            shutil.copy2(sf, GLOBAL_STREAM / f"{sid}.csv")
            copied_s += 1
        if ff.exists():
            shutil.copy2(ff, GLOBAL_FORCE / f"{sid}.csv")
            copied_f += 1

# Write merged station list
with open(GLOBAL_DIR / "station_list.txt", "w") as f:
    f.write("\n".join(sorted(seen_stations)))

# 4. Merge attributes from all 4 clusters into one global attributes.csv
import pandas as pd
attr_dfs = []
for cl in [1, 2, 3, 4]:
    attr_path = WORK_DATA_ROOT / f"Cluster_{cl}" / "attributes.csv"
    if attr_path.exists():
        df = pd.read_csv(attr_path)
        df["source_cluster"] = cl
        attr_dfs.append(df)
    else:
        print(f"  WARNING: No attributes.csv found for Cluster_{cl}")

if attr_dfs:
    global_attrs = pd.concat(attr_dfs, ignore_index=True)
    # Remove duplicates if any station appears in multiple clusters
    id_col = "gauge_id" if "gauge_id" in global_attrs.columns else global_attrs.columns[0]
    global_attrs = global_attrs.drop_duplicates(subset=[id_col])
    global_attrs.to_csv(GLOBAL_DIR / "attributes.csv", index=False)
    print(f"  Global attributes.csv: {len(global_attrs)} basins")

print(f"\n✓ Cluster_0 (global) created in {GLOBAL_DIR}")
print(f"  Streamflow files : {copied_s}")
print(f"  Forcing files    : {copied_f}")
print(f"  Total basins     : {len(seen_stations)}")
print(f"  Duplicate skips  : {len(skipped)}")
if skipped:
    print(f"  Skipped stations : {skipped[:5]}{'...' if len(skipped)>5 else ''}")

# 5. Update DATA_ROOT to writable location
DATA_ROOT = WORK_DATA_ROOT
print(f"\nDATA_ROOT updated to: {DATA_ROOT}")
print("Run Cell 5 next to set cfg.DATA_ROOT to this path.")


  Copied root file: feature_config.json
  Cluster_1 copied: 98 files → /kaggle/working/prepared_data_global/Cluster_1
  Cluster_2 copied: 102 files → /kaggle/working/prepared_data_global/Cluster_2
  Cluster_3 copied: 132 files → /kaggle/working/prepared_data_global/Cluster_3
  Cluster_4 copied: 92 files → /kaggle/working/prepared_data_global/Cluster_4
  Global attributes.csv: 206 basins

✓ Cluster_0 (global) created in /kaggle/working/prepared_data_global/Cluster_0
  Streamflow files : 206
  Forcing files    : 206
  Total basins     : 206
  Duplicate skips  : 0

DATA_ROOT updated to: /kaggle/working/prepared_data_global
Run Cell 5 next to set cfg.DATA_ROOT to this path.


In [2]:
# ── Cell 2: Update DATA_ROOT to writable working directory ───────────────────
# This cell must run AFTER Cell 1 which builds the writable data structure.
# It updates DATA_ROOT so all subsequent cells use the writable copy.

from pathlib import Path
import shutil

WORK_DATA_ROOT = Path("/kaggle/working/prepared_data_global")

if not WORK_DATA_ROOT.exists():
    raise RuntimeError(
        "WORK_DATA_ROOT not found. Run Cell 1 first to build the writable data structure.")

# Verify Cluster_0 exists and is populated
global_dir = WORK_DATA_ROOT / "Cluster_0"
if not global_dir.exists():
    raise RuntimeError("Cluster_0 not found in WORK_DATA_ROOT. Re-run Cell 1.")

n_stream = len(list((global_dir / "streamflow").glob("*.csv")))
n_force  = len(list((global_dir / "forcing").glob("*.csv")))
print(f"Cluster_0 validation:")
print(f"  Streamflow files : {n_stream}")
print(f"  Forcing files    : {n_force}")
print(f"  Station list     : {(global_dir / 'station_list.txt').exists()}")
print(f"  Attributes CSV   : {(global_dir / 'attributes.csv').exists()}")

if n_stream == 0:
    raise RuntimeError("No streamflow files in Cluster_0. Re-run Cell 1.")

# Update DATA_ROOT — this will be picked up by Cell 5 config
DATA_ROOT = WORK_DATA_ROOT
print(f"\n✓ DATA_ROOT set to: {DATA_ROOT}")


Cluster_0 validation:
  Streamflow files : 206
  Forcing files    : 206
  Station list     : True
  Attributes CSV   : True

✓ DATA_ROOT set to: /kaggle/working/prepared_data_global


In [3]:
# ── Cell 3: Validate global cluster structure ─────────────────────────────────
# Recreate loader with updated DATA_ROOT pointing to writable working directory

from pathlib import Path

# Ensure DATA_ROOT is the writable working copy
WORK_DATA_ROOT = Path("/kaggle/working/prepared_data_global")
DATA_ROOT = WORK_DATA_ROOT

# Reload cfg with correct DATA_ROOT
# (cfg is defined in Cell 9 — we just update the path here so Cell 5 picks it up)
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Cluster_0 exists: {(DATA_ROOT / 'Cluster_0').exists()}")

# Quick file count validation
for cl in [0, 1, 2, 3, 4]:
    cl_dir = DATA_ROOT / f"Cluster_{cl}"
    if cl_dir.exists():
        sf = cl_dir / "streamflow"
        fo = cl_dir / "forcing"
        ns = len(list(sf.glob("*.csv"))) if sf.exists() else 0
        nf = len(list(fo.glob("*.csv"))) if fo.exists() else 0
        attrs = (cl_dir / "attributes.csv").exists()
        slist = (cl_dir / "station_list.txt").exists()
        label = "GLOBAL" if cl==0 else f"Cluster_{cl}"
        print(f"  {label:12s}: streamflow={ns:3d}  forcing={nf:3d}  "
              f"attrs={'✓' if attrs else '✗'}  station_list={'✓' if slist else '✗'}")
    else:
        print(f"  Cluster_{cl}: NOT FOUND")

print("\nValidation complete. Proceed to Cell 4 (install) then Cell 5 (config).")


DATA_ROOT: /kaggle/working/prepared_data_global
Cluster_0 exists: True
  GLOBAL      : streamflow=206  forcing=206  attrs=✓  station_list=✓
  Cluster_1   : streamflow= 46  forcing= 46  attrs=✓  station_list=✓
  Cluster_2   : streamflow= 50  forcing= 50  attrs=✓  station_list=✓
  Cluster_3   : streamflow= 65  forcing= 65  attrs=✓  station_list=✓
  Cluster_4   : streamflow= 45  forcing= 45  attrs=✓  station_list=✓

Validation complete. Proceed to Cell 4 (install) then Cell 5 (config).


In [4]:
# Install PyTorch Lightning only when it is absent.
# Kaggle normally already contains PyTorch, NumPy, pandas, scikit-learn and matplotlib.
import importlib.util, subprocess, sys

if importlib.util.find_spec("pytorch_lightning") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning>=2.2,<3"])

print("Environment setup complete.")

Environment setup complete.


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# EDIT ONLY THIS CELL
# ══════════════════════════════════════════════════════════════════════════════

PLATFORM = "kaggle"
DATA_ROOT = None
# DATA_ROOT must point to the WRITABLE working copy created in Cell 1+2
# NOT to /kaggle/input/ which is read-only
DATA_ROOT = Path("/kaggle/working/prepared_data_global")

# ── What to train ─────────────────────────────────────────────────────────────
# For GLOBAL baseline: set CLUSTERS_TO_TRAIN = [0]  (Cluster_0 = all basins)
# For CLUSTER models:  set CLUSTERS_TO_TRAIN = [1,2,3,4]
CLUSTERS_TO_TRAIN = [0]          # ← [0] for global TFT baseline

# ── Training config ───────────────────────────────────────────────────────────
MAX_EPOCHS  = 60                 # 60 fits in 12-hr Kaggle session for global model
BATCH_SIZE  = 128                # 128 on single T4 (16GB) — safe for 365-day lookback
NUM_WORKERS = 4
PRECISION   = "16-mixed"         # faster on T4; safe after NaN fix
RANDOM_SEED = 42
LOSS_BETA   = 2.0                # 2.0 = primary model; 4.0 = peak-focus sensitivity

# ── Checkpoint resume (cross-session training) ────────────────────────────────
# If your Kaggle session timed out mid-training, upload the checkpoint as a
# dataset and set this path. Set to None for fresh training.
RESUME_FROM_CHECKPOINT = None
# Example: RESUME_FROM_CHECKPOINT = "/kaggle/input/tft-global-ckpt/best.ckpt"

# ── Run tag ───────────────────────────────────────────────────────────────────
# Use different tags to distinguish experiments in output filenames
RUN_TAG = "TFT_global"           # "TFT_global" / "TFT_cluster" / "TFT_beta4"

# ── Data periods ──────────────────────────────────────────────────────────────
TRAIN_START, TRAIN_END = "1986-01-01", "2005-12-31"
VAL_START,   VAL_END   = "2006-01-01", "2012-12-31"
TEST_START,  TEST_END  = "2013-01-01", "2020-12-31"

# ── Sequence config ───────────────────────────────────────────────────────────
LOOKBACK         = 365
FORECAST_HORIZON = 30
LEAD_TIMES       = [1, 3, 7, 14, 30]
SAMPLE_STRIDE    = 1
RESET_CLUSTER_OUTPUT = True

print("Configuration loaded.")
print(f"  Clusters to train : {CLUSTERS_TO_TRAIN}")
print(f"  Max epochs        : {MAX_EPOCHS}")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Loss beta         : {LOSS_BETA}")
print(f"  Run tag           : {RUN_TAG}")
print(f"  Resume checkpoint : {RESUME_FROM_CHECKPOINT}")


Configuration loaded.
  Clusters to train : [0]
  Max epochs        : 60
  Batch size        : 128
  Loss beta         : 2.0
  Run tag           : TFT_global
  Resume checkpoint : None


In [6]:
DATA_ROOT

PosixPath('/kaggle/working/prepared_data_global')

In [7]:
# Imports, reproducibility and device information
import os
import gc
import json
import math
import pickle
import random
import shutil
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

warnings.filterwarnings("ignore")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.set_float32_matmul_precision("high")
pl.seed_everything(RANDOM_SEED, workers=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Lightning:", pl.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPU is not enabled. Kaggle training will be much slower on CPU.")

Seed set to 42


PyTorch: 2.10.0+cu128
Lightning: 2.6.4
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [8]:
# Resolve the input and output paths

def resolve_data_root(user_path):
    if user_path is not None:
        root = Path(user_path)
        if not root.exists():
            raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
        if not (root / "feature_config.json").exists():
            raise FileNotFoundError(f"feature_config.json not found inside: {root}")
        return root

    search_base = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    candidates = sorted({p.parent.resolve() for p in search_base.rglob("feature_config.json")})

    if len(candidates) == 1:
        print("Automatically detected DATA_ROOT:", candidates[0])
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No feature_config.json was found. Add the prepared dataset to Kaggle "
            "or set DATA_ROOT explicitly."
        )

    print("Multiple prepared-data folders were found:")
    for candidate in candidates:
        print(" -", candidate)
    raise RuntimeError("Set DATA_ROOT explicitly to one of the paths printed above.")

DATA_ROOT = resolve_data_root(DATA_ROOT)
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = WORK_ROOT / "tft_beta4_corrected_outputs"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_ROOT: /kaggle/working/prepared_data_global
OUTPUT_DIR: /kaggle/working/tft_beta4_corrected_outputs


In [9]:
@dataclass
class Config:
    DATA_ROOT: Path = DATA_ROOT
    OUTPUT_DIR: Path = OUTPUT_DIR
    CKPT_DIR: Path = CKPT_DIR

    TRAIN_START: str = TRAIN_START
    TRAIN_END:   str = TRAIN_END
    VAL_START:   str = VAL_START
    VAL_END:     str = VAL_END
    TEST_START:  str = TEST_START
    TEST_END:    str = TEST_END

    LOOKBACK:          int  = LOOKBACK
    FORECAST_HORIZON:  int  = FORECAST_HORIZON
    LEAD_TIMES:        list = field(default_factory=lambda: LEAD_TIMES.copy())
    SAMPLE_STRIDE:     int  = SAMPLE_STRIDE
    QUANTILES:         list = field(default_factory=lambda: [0.1, 0.5, 0.9])

    TFT_HIDDEN: int   = 128      # increased from 64 for global model capacity
    TFT_HEADS:  int   = 4
    TFT_DROP:   float = 0.2
    TFT_GRN:    int   = 64

    BATCH_SIZE:  int   = BATCH_SIZE
    NUM_WORKERS: int   = NUM_WORKERS
    LR:          float = 1e-3
    LR_MIN:      float = 1e-5
    MAX_EPOCHS:  int   = MAX_EPOCHS
    PATIENCE:    int   = max(5, MAX_EPOCHS // 6)   # ~10 for 60 epochs
    GRAD_CLIP:   float = 1.0
    LOSS_BETA:   float = LOSS_BETA
    W_NSE:       float = 0.7
    W_Q:         float = 0.3
    PRECISION:   str   = PRECISION

    RESUME_FROM_CHECKPOINT: object = RESUME_FROM_CHECKPOINT
    RUN_TAG:               str    = RUN_TAG

    DYNAMIC_PAST: list = field(
        default_factory=lambda: ["streamflow", "precip", "tmax", "tmin", "pet"]
    )
    DYNAMIC_FUTURE: list = field(
        default_factory=lambda: ["precip", "tmax", "tmin", "pet"]
    )
    CLUSTERS: list = field(default_factory=lambda: CLUSTERS_TO_TRAIN.copy())

cfg = Config()

assert cfg.FORECAST_HORIZON >= max(cfg.LEAD_TIMES)
assert cfg.TFT_HIDDEN % cfg.TFT_HEADS == 0
print(cfg)


Config(DATA_ROOT=PosixPath('/kaggle/working/prepared_data_global'), OUTPUT_DIR=PosixPath('/kaggle/working/tft_beta4_corrected_outputs'), CKPT_DIR=PosixPath('/kaggle/working/tft_beta4_corrected_outputs/checkpoints'), TRAIN_START='1986-01-01', TRAIN_END='2005-12-31', VAL_START='2006-01-01', VAL_END='2012-12-31', TEST_START='2013-01-01', TEST_END='2020-12-31', LOOKBACK=365, FORECAST_HORIZON=30, LEAD_TIMES=[1, 3, 7, 14, 30], SAMPLE_STRIDE=1, QUANTILES=[0.1, 0.5, 0.9], TFT_HIDDEN=128, TFT_HEADS=4, TFT_DROP=0.2, TFT_GRN=64, BATCH_SIZE=128, NUM_WORKERS=4, LR=0.001, LR_MIN=1e-05, MAX_EPOCHS=60, PATIENCE=10, GRAD_CLIP=1.0, LOSS_BETA=2.0, W_NSE=0.7, W_Q=0.3, PRECISION='16-mixed', RESUME_FROM_CHECKPOINT=None, RUN_TAG='TFT_global', DYNAMIC_PAST=['streamflow', 'precip', 'tmax', 'tmin', 'pet'], DYNAMIC_FUTURE=['precip', 'tmax', 'tmin', 'pet'], CLUSTERS=[0])


In [10]:
# Universal data loader with robust date and column-name handling

FORCING_ALIASES = {
    "prcp(mm/day)": "precip",
    "prcp": "precip",
    "precipitation": "precip",
    "precip": "precip",
    "tmax(c)": "tmax",
    "tmax": "tmax",
    "tmin(c)": "tmin",
    "tmin": "tmin",
    "pet(mm/day)": "pet",
    "pet": "pet",
}

FLOW_ALIASES = {
    "streamflow": "streamflow",
    "streamflow(m3/s)": "streamflow",
    "discharge": "streamflow",
    "discharge(m3/s)": "streamflow",
    "flow": "streamflow",
    "q": "streamflow",
}


def normalized_name(name):
    return str(name).strip().lower().replace(" ", "")


def rename_known_columns(df, aliases):
    lookup = {normalized_name(k): v for k, v in aliases.items()}
    rename = {}
    for col in df.columns:
        key = normalized_name(col)
        if key in lookup:
            rename[col] = lookup[key]
    return df.rename(columns=rename)


def parse_date_index(df):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    if all(k in lower_map for k in ["year", "month", "day"]):
        date = pd.to_datetime({
            "year": pd.to_numeric(df[lower_map["year"]], errors="coerce"),
            "month": pd.to_numeric(df[lower_map["month"]], errors="coerce"),
            "day": pd.to_numeric(df[lower_map["day"]], errors="coerce"),
        }, errors="coerce")
        drop_cols = [lower_map[k] for k in ["year", "month", "day"]]
        df = df.drop(columns=drop_cols)
    else:
        date_col = next((lower_map[k] for k in ["date", "datetime", "time"] if k in lower_map), None)
        if date_col is None:
            raise KeyError(f"No date column found. Available columns: {list(df.columns)}")
        # Try day-first first because the original streamflow files use that format.
        date = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        if date.isna().mean() > 0.5:
            date = pd.to_datetime(df[date_col], errors="coerce")
        df = df.drop(columns=[date_col])

    df = df.copy()
    df.index = date
    df.index.name = "date"
    df = df.loc[~df.index.isna()].sort_index()
    df = df.loc[~df.index.duplicated(keep="last")]
    return df


class UniversalDataLoader:
    def __init__(self, cfg):
        self.cfg = cfg
        feature_path = cfg.DATA_ROOT / "feature_config.json"
        with open(feature_path, "r", encoding="utf-8") as handle:
            feature_cfg = json.load(handle)
        self.static_cols = list(feature_cfg["static_features"])
        print(f"Static features requested: {len(self.static_cols)}")

    def cluster_path(self, cluster_id, *parts):
        return self.cfg.DATA_ROOT / f"Cluster_{cluster_id}" / Path(*parts)

    def validate_cluster_structure(self, cluster_id):
        required = [
            self.cluster_path(cluster_id, "attributes.csv"),
            self.cluster_path(cluster_id, "station_list.txt"),
            self.cluster_path(cluster_id, "streamflow"),
            self.cluster_path(cluster_id, "forcing"),
        ]
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise FileNotFoundError("Missing required cluster items:\n" + "\n".join(missing))

    def load_attributes(self, cluster_id):
        path = self.cluster_path(cluster_id, "attributes.csv")
        raw = pd.read_csv(path)
        gauge_col = next((c for c in raw.columns if str(c).strip().lower() in {"gauge_id", "station", "station_id"}), None)
        if gauge_col is None:
            raise KeyError(f"Gauge identifier column not found in {path}")
        raw[gauge_col] = raw[gauge_col].astype(str)
        raw = raw.set_index(gauge_col)

        available = [c for c in self.static_cols if c in raw.columns]
        missing = [c for c in self.static_cols if c not in raw.columns]
        if missing:
            print(f"WARNING: Cluster {cluster_id} is missing {len(missing)} configured static features.")
        if not available:
            raise ValueError(f"No configured static features were found in {path}")

        attrs = raw[available].apply(pd.to_numeric, errors="coerce")
        attrs = attrs.replace([np.inf, -np.inf], np.nan)
        attrs.index = attrs.index.astype(str)
        return attrs

    def load_station_list(self, cluster_id):
        path = self.cluster_path(cluster_id, "station_list.txt")
        with open(path, "r", encoding="utf-8") as handle:
            return [line.strip() for line in handle if line.strip()]

    def load_basin(self, cluster_id, gauge_id):
        gid = str(gauge_id)
        q_path = self.cluster_path(cluster_id, "streamflow", f"{gid}.csv")
        f_path = self.cluster_path(cluster_id, "forcing", f"{gid}.csv")
        if not q_path.exists() or not f_path.exists():
            return None

        q = parse_date_index(pd.read_csv(q_path))
        q = rename_known_columns(q, FLOW_ALIASES)
        if "streamflow" not in q.columns:
            raise KeyError(f"No streamflow column recognized in {q_path}. Columns: {list(q.columns)}")
        q = q[["streamflow"]].apply(pd.to_numeric, errors="coerce")

        forcing = parse_date_index(pd.read_csv(f_path))
        forcing = rename_known_columns(forcing, FORCING_ALIASES)
        required_forcing = list(dict.fromkeys(cfg.DYNAMIC_FUTURE))
        missing_forcing = [c for c in required_forcing if c not in forcing.columns]
        if missing_forcing:
            raise KeyError(
                f"Missing forcing variables {missing_forcing} in {f_path}. "
                f"Columns after aliasing: {list(forcing.columns)}"
            )
        forcing = forcing[required_forcing].apply(pd.to_numeric, errors="coerce")

        df = q.join(forcing, how="inner").sort_index()
        if df.empty:
            return None

        # Reindex daily so each model step represents one day.
        full_index = pd.date_range(df.index.min(), df.index.max(), freq="D")
        df = df.reindex(full_index)
        df.index.name = "date"

        # Negative and extreme streamflow values are invalid.
        df.loc[df["streamflow"] < 0, "streamflow"] = np.nan
        valid_q = df["streamflow"].dropna()
        if len(valid_q) >= 100:
            q99 = valid_q.quantile(0.99)
            if np.isfinite(q99) and q99 > 0:
                df.loc[df["streamflow"] > 5.0 * q99, "streamflow"] = np.nan

        # Preserve the original-observation mask before short-gap interpolation.
        df["obs_mask"] = df["streamflow"].notna()
        df["streamflow"] = df["streamflow"].interpolate(
            method="linear", limit=7, limit_area="inside"
        )
        return df

    def load_all_basins(self, cluster_id, tiers_allowed=None, quality_df=None):
        self.validate_cluster_structure(cluster_id)
        gauge_ids = self.load_station_list(cluster_id)

        if quality_df is not None and tiers_allowed is not None:
            quality_df = quality_df.copy()
            quality_df.index = quality_df.index.astype(str)
            gauge_ids = [
                gid for gid in gauge_ids
                if str(gid) in quality_df.index
                and str(quality_df.loc[str(gid), "Tier"]) in tiers_allowed
            ]

        data = {}
        failures = []
        for gid in tqdm(gauge_ids, desc=f"Loading Cluster {cluster_id}"):
            try:
                basin = self.load_basin(cluster_id, gid)
                if basin is not None and len(basin) >= cfg.LOOKBACK + cfg.FORECAST_HORIZON:
                    data[str(gid)] = basin
            except Exception as exc:
                failures.append((str(gid), str(exc)))

        print(f"Cluster {cluster_id}: loaded {len(data)}/{len(gauge_ids)} basins")
        if failures:
            print(f"WARNING: {len(failures)} basins failed. First three failures:")
            for item in failures[:3]:
                print(" ", item)
        return data

loader = UniversalDataLoader(cfg)
print("Data loader ready.")

Static features requested: 54
Data loader ready.


In [11]:
# Validate one example basin before training
first_cluster = cfg.CLUSTERS[0]
loader.validate_cluster_structure(first_cluster)
example_station = loader.load_station_list(first_cluster)[0]
example_df = loader.load_basin(first_cluster, example_station)

print("Example station:", example_station)
print("Date range:", example_df.index.min(), "to", example_df.index.max())
print("Columns:", list(example_df.columns))
print("Shape:", example_df.shape)
print("Observed streamflow fraction:", float(example_df["obs_mask"].mean()))
print(example_df.head())

Example station: 10004
Date range: 1980-01-01 00:00:00 to 2018-12-31 00:00:00
Columns: ['streamflow', 'precip', 'tmax', 'tmin', 'pet', 'obs_mask']
Shape: (14245, 6)
Observed streamflow fraction: 0.5955071955071956
            streamflow  precip   tmax   tmin  pet  obs_mask
date                                                       
1980-01-01         NaN     0.0  27.03  13.95  NaN     False
1980-01-02         NaN     0.0  25.76  10.64  NaN     False
1980-01-03         NaN     0.0  25.39  10.73  NaN     False
1980-01-04         NaN     0.0  24.60  10.91  NaN     False
1980-01-05         NaN     0.0  24.58  11.60  NaN     False


In [12]:
# Scaling and masked losses

class HydroScaler:
    def __init__(self):
        self.q_mean = {}
        self.q_std = {}
        self.dynamic_scalers = {}
        self.dynamic_fill = {}
        self.attr_scaler = StandardScaler()
        self.attr_fill = None
        self.attr_columns = None

    def fit(self, basin_data, attributes_df, cfg):
        dynamic = list(dict.fromkeys(
            c for c in cfg.DYNAMIC_PAST + cfg.DYNAMIC_FUTURE if c != "streamflow"
        ))

        for bid, df in basin_data.items():
            train = df.loc[cfg.TRAIN_START:cfg.TRAIN_END].copy()
            observed = train.loc[train["obs_mask"].fillna(False), "streamflow"].dropna()
            if len(observed) < 10:
                observed = train["streamflow"].dropna()
            if len(observed) < 2:
                raise ValueError(f"Insufficient training streamflow observations for basin {bid}")

            log_q = np.log1p(observed.clip(lower=0).to_numpy(dtype=float))
            self.q_mean[bid] = float(np.mean(log_q))
            self.q_std[bid] = float(max(np.std(log_q), 1e-6))

            missing = [c for c in dynamic if c not in train.columns]
            if missing:
                raise KeyError(f"Basin {bid} is missing dynamic variables: {missing}")

            x = train[dynamic].replace([np.inf, -np.inf], np.nan)
            fill = x.median(numeric_only=True).reindex(dynamic).fillna(0.0)
            scaler = StandardScaler().fit(x.fillna(fill))
            self.dynamic_scalers[bid] = (scaler, dynamic)
            self.dynamic_fill[bid] = fill

        aligned_attrs = attributes_df.reindex(list(basin_data.keys())).copy()
        aligned_attrs = aligned_attrs.replace([np.inf, -np.inf], np.nan)
        self.attr_columns = list(aligned_attrs.columns)
        self.attr_fill = aligned_attrs.median(numeric_only=True).reindex(self.attr_columns).fillna(0.0)
        self.attr_scaler.fit(aligned_attrs[self.attr_columns].fillna(self.attr_fill))
        return self

    def transform_q(self, bid, q):
        q = np.asarray(q, dtype=float)
        replacement = np.expm1(self.q_mean[bid])
        q = np.where(np.isfinite(q), q, replacement)
        return (np.log1p(np.clip(q, 0, None)) - self.q_mean[bid]) / self.q_std[bid]

    def inverse_q(self, bid, q_scaled):
        q_scaled = np.asarray(q_scaled, dtype=float)
        q = np.expm1(q_scaled * self.q_std[bid] + self.q_mean[bid])
        return np.clip(q, 0.0, None)

    def transform_dynamic(self, bid, df):
        out = df.copy()
        out["streamflow_sc"] = self.transform_q(bid, out["streamflow"].to_numpy())
        scaler, columns = self.dynamic_scalers[bid]
        fill = self.dynamic_fill[bid]
        x = out[columns].replace([np.inf, -np.inf], np.nan).fillna(fill)
        out.loc[:, columns] = scaler.transform(x)
        return out

    def transform_static(self, attributes_df, basin_ids):
        aligned = attributes_df.reindex(basin_ids)[self.attr_columns]
        aligned = aligned.replace([np.inf, -np.inf], np.nan).fillna(self.attr_fill)
        result = self.attr_scaler.transform(aligned)
        if not np.isfinite(result).all():
            raise FloatingPointError("Static scaling produced non-finite values.")
        return result.astype(np.float32)


def masked_weighted_nse_loss(pred, target, mask, beta=2.0, eps=1e-6):
    valid = mask.bool() & torch.isfinite(pred) & torch.isfinite(target)
    if valid.sum() < 2:
        return pred.sum() * 0.0
    p = pred[valid]
    t = target[valid]
    weights = (t.abs() + eps).pow(beta)
    weights = weights / weights.mean().clamp_min(eps)
    weighted_mean = (weights * t).sum() / weights.sum().clamp_min(eps)
    numerator = (weights * (p - t).pow(2)).sum()
    denominator = (weights * (t - weighted_mean).pow(2)).sum()
    if denominator.detach().item() <= eps:
        return (weights * (p - t).pow(2)).mean()
    return numerator / denominator.clamp_min(eps)


def masked_quantile_loss(quantile_pred, target, mask, quantiles):
    valid = mask.bool() & torch.isfinite(target)
    if valid.sum() == 0:
        return quantile_pred.sum() * 0.0
    errors = target.unsqueeze(-1) - quantile_pred
    q = torch.as_tensor(quantiles, dtype=quantile_pred.dtype, device=quantile_pred.device)
    pinball = torch.maximum(q * errors, (q - 1.0) * errors)
    expanded_mask = valid.unsqueeze(-1).expand_as(pinball)
    return pinball[expanded_mask].mean()


def combined_loss(pred, quantile_pred, target, mask, cfg):
    nse_component = masked_weighted_nse_loss(pred, target, mask, cfg.LOSS_BETA)
    quantile_component = masked_quantile_loss(quantile_pred, target, mask, cfg.QUANTILES)
    loss = cfg.W_NSE * nse_component + cfg.W_Q * quantile_component
    if not torch.isfinite(loss):
        raise FloatingPointError("Loss became non-finite.")
    return loss

print("Scaler and losses defined.")

Scaler and losses defined.


In [13]:
# Memory-efficient indexed dataset

class BasinTensorStore:
    def __init__(self, basin_data, attributes_df, scaler, cfg):
        self.data = {}
        basin_ids = list(basin_data.keys())
        static_matrix = scaler.transform_static(attributes_df, basin_ids)
        static_map = {bid: static_matrix[i] for i, bid in enumerate(basin_ids)}

        past_columns = ["streamflow_sc" if c == "streamflow" else c for c in cfg.DYNAMIC_PAST]
        future_columns = list(cfg.DYNAMIC_FUTURE)

        for bid in basin_ids:
            transformed = scaler.transform_dynamic(bid, basin_data[bid])
            missing_past = [c for c in past_columns if c not in transformed.columns]
            missing_future = [c for c in future_columns if c not in transformed.columns]
            if missing_past or missing_future:
                raise KeyError(
                    f"Basin {bid}: missing past={missing_past}, future={missing_future}"
                )

            record = {
                "dates": transformed.index.to_numpy(),
                "past": transformed[past_columns].to_numpy(np.float32),
                "future": transformed[future_columns].to_numpy(np.float32),
                "target": transformed["streamflow_sc"].to_numpy(np.float32),
                "mask": transformed["obs_mask"].fillna(False).to_numpy(bool),
                "static": static_map[bid],
            }
            for key in ["past", "future", "target", "static"]:
                if not np.isfinite(record[key]).all():
                    raise FloatingPointError(f"Non-finite values in {key} for basin {bid}")
            self.data[bid] = record


class IndexedHydroDataset(Dataset):
    def __init__(self, store, cfg, start_date, end_date):
        self.store = store
        self.cfg = cfg
        self.origins = []
        start = np.datetime64(start_date)
        end = np.datetime64(end_date)

        for bid, record in store.data.items():
            dates = record["dates"]
            n = len(dates)
            for origin in range(cfg.LOOKBACK, n - cfg.FORECAST_HORIZON + 1, cfg.SAMPLE_STRIDE):
                forecast_start = dates[origin]
                forecast_end = dates[origin + cfg.FORECAST_HORIZON - 1]
                if forecast_start >= start and forecast_end <= end:
                    self.origins.append((bid, origin))

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, index):
        bid, origin = self.origins[index]
        record = self.store.data[bid]
        h = self.cfg.FORECAST_HORIZON
        l = self.cfg.LOOKBACK
        return {
            "x_past": torch.from_numpy(record["past"][origin-l:origin]),
            "x_future": torch.from_numpy(record["future"][origin:origin+h]),
            "x_static": torch.from_numpy(record["static"]),
            "y": torch.from_numpy(record["target"][origin:origin+h]),
            "mask": torch.from_numpy(record["mask"][origin:origin+h]),
            "basin_id": bid,
            "origin": origin,
        }


def make_dataloaders(basin_data, attributes_df, scaler, cfg):
    store = BasinTensorStore(basin_data, attributes_df, scaler, cfg)
    train_ds = IndexedHydroDataset(store, cfg, cfg.TRAIN_START, cfg.TRAIN_END)
    val_ds = IndexedHydroDataset(store, cfg, cfg.VAL_START, cfg.VAL_END)
    test_ds = IndexedHydroDataset(store, cfg, cfg.TEST_START, cfg.TEST_END)

    if min(len(train_ds), len(val_ds), len(test_ds)) == 0:
        raise ValueError(
            f"At least one split is empty: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}. "
            "Check date coverage and configured periods."
        )

    common = dict(
        batch_size=cfg.BATCH_SIZE,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=cfg.NUM_WORKERS > 0,
    )
    train_dl = DataLoader(train_ds, shuffle=True, **common)
    val_dl = DataLoader(val_ds, shuffle=False, **common)
    test_dl = DataLoader(test_ds, shuffle=False, **common)
    print(f"Samples: train={len(train_ds):,}, val={len(val_ds):,}, test={len(test_ds):,}")
    return train_dl, val_dl, test_dl

print("Dataset classes defined.")

Dataset classes defined.


In [14]:
# Corrected TFT model

class GLU(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.0):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)
        self.gate = nn.Linear(input_size, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.linear(x)) * torch.sigmoid(self.gate(x))


class GRN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, context_size=None, dropout=0.0):
        super().__init__()
        output_size = output_size or hidden_size
        self.fc1 = nn.Linear(input_size + (context_size or 0), hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.glu = GLU(hidden_size, output_size, dropout)
        self.norm = nn.LayerNorm(output_size)
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()

    def forward(self, x, context=None):
        joined = torch.cat([x, context], dim=-1) if context is not None else x
        hidden = self.fc2(F.elu(self.fc1(joined)))
        return self.norm(self.skip(x) + self.glu(hidden))


class VariableSelectionNetwork(nn.Module):
    def __init__(self, n_variables, hidden_size, context_size=None, dropout=0.0):
        super().__init__()
        if n_variables < 1:
            raise ValueError("VariableSelectionNetwork requires at least one variable.")
        self.variable_grns = nn.ModuleList([
            GRN(1, hidden_size, hidden_size, dropout=dropout)
            for _ in range(n_variables)
        ])
        self.weight_grn = GRN(
            n_variables, hidden_size, n_variables,
            context_size=context_size, dropout=dropout
        )

    def forward(self, x, context=None):
        temporal = x.dim() == 3
        if not temporal:
            x = x.unsqueeze(1)
        batch, steps, variables = x.shape
        encoded = torch.stack(
            [self.variable_grns[i](x[..., i:i+1]) for i in range(variables)],
            dim=-2,
        )
        flat = x.reshape(batch, steps, variables)
        if context is not None and context.dim() == 2:
            context = context.unsqueeze(1).expand(-1, steps, -1)
        weights = torch.softmax(self.weight_grn(flat, context), dim=-1)
        selected = (encoded * weights.unsqueeze(-1)).sum(dim=-2)
        if not temporal:
            selected = selected.squeeze(1)
            weights = weights.squeeze(1)
        return selected, weights


class ClusterTFT(nn.Module):
    def __init__(self, n_past, n_future, n_static, hidden=64, heads=4,
                 horizon=30, n_quantiles=3, dropout=0.2, grn_hidden=64):
        super().__init__()
        self.horizon = horizon
        self.n_quantiles = n_quantiles

        self.static_projection = nn.Linear(n_static, hidden)
        self.static_selection_context = GRN(hidden, grn_hidden, hidden, dropout=dropout)
        self.static_enrichment_context = GRN(hidden, grn_hidden, hidden, dropout=dropout)
        self.static_hidden = GRN(hidden, grn_hidden, hidden, dropout=dropout)
        self.static_cell = GRN(hidden, grn_hidden, hidden, dropout=dropout)

        self.past_vsn = VariableSelectionNetwork(n_past, hidden, hidden, dropout)
        self.future_vsn = VariableSelectionNetwork(n_future, hidden, hidden, dropout)

        self.encoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.decoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.enrichment = GRN(hidden, grn_hidden, hidden, hidden, dropout)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attention_norm = nn.LayerNorm(hidden)
        self.feed_forward = GRN(hidden, grn_hidden, hidden, dropout=dropout)
        self.output_norm = nn.LayerNorm(hidden)
        self.quantile_heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_quantiles)])

    @staticmethod
    def build_attention_mask(encoder_length, total_length, device):
        # Boolean mask: True means "do not attend".
        # This avoids the invalid 0 * -inf operation that caused all predictions to be NaN.
        mask = torch.zeros((total_length, total_length), dtype=torch.bool, device=device)
        decoder_length = total_length - encoder_length
        if decoder_length > 0:
            causal_decoder_mask = torch.triu(
                torch.ones((decoder_length, decoder_length), dtype=torch.bool, device=device),
                diagonal=1,
            )
            mask[encoder_length:, encoder_length:] = causal_decoder_mask
        return mask

    def forward(self, x_past, x_future, x_static, return_attention=False):
        static = self.static_projection(x_static)
        selection_context = self.static_selection_context(static)
        enrichment_context = self.static_enrichment_context(static)
        hidden_state = self.static_hidden(static).unsqueeze(0)
        cell_state = self.static_cell(static).unsqueeze(0)

        past_selected, past_weights = self.past_vsn(x_past, selection_context)
        future_selected, future_weights = self.future_vsn(x_future, selection_context)

        encoded, (h_n, c_n) = self.encoder(past_selected, (hidden_state, cell_state))
        decoded, _ = self.decoder(future_selected, (h_n, c_n))
        encoded = self.dropout(encoded)
        decoded = self.dropout(decoded)

        encoder_length = encoded.size(1)
        combined = torch.cat([encoded, decoded], dim=1)
        total_length = combined.size(1)
        context = enrichment_context.unsqueeze(1).expand(-1, total_length, -1)
        enriched = self.enrichment(combined, context)

        attention_mask = self.build_attention_mask(encoder_length, total_length, x_past.device)
        attended, attention_weights = self.attention(
            enriched, enriched, enriched,
            attn_mask=attention_mask,
            need_weights=return_attention,
        )

        attended = self.attention_norm(enriched + attended)
        output = self.output_norm(attended + self.feed_forward(attended))
        future_output = output[:, encoder_length:, :]

        quantile_predictions = torch.stack(
            [head(future_output).squeeze(-1) for head in self.quantile_heads],
            dim=-1,
        )
        result = {
            "pred": quantile_predictions[..., self.n_quantiles // 2],
            "quantile_preds": quantile_predictions,
            "past_vsn_weights": past_weights,
            "future_vsn_weights": future_weights,
        }
        if return_attention:
            result["attn_weights"] = attention_weights
            result["attn_mask"] = attention_mask
        return result

print("Corrected ClusterTFT defined.")

Corrected ClusterTFT defined.


In [15]:
# Lightning wrapper with fail-fast finite-value checks

class Forecaster(pl.LightningModule):
    def __init__(self, model, cfg, cluster_id):
        super().__init__()
        self.model      = model
        self.cfg        = cfg
        self.cluster_id = str(cluster_id)
        self.save_hyperparameters({"cluster_id": self.cluster_id})

    def forward(self, batch):
        return self.model(batch["x_past"], batch["x_future"], batch["x_static"])

    @staticmethod
    def assert_finite_tensor(name, tensor):
        if not torch.isfinite(tensor).all():
            nan_count = torch.isnan(tensor).sum().item()
            inf_count = torch.isinf(tensor).sum().item()
            raise FloatingPointError(
                f"{name} contains NaN={nan_count}, Inf={inf_count}")

    def shared_step(self, batch, stage):
        for name in ["x_past", "x_future", "x_static", "y"]:
            self.assert_finite_tensor(name, batch[name])
        output = self(batch)
        self.assert_finite_tensor("prediction",          output["pred"])
        self.assert_finite_tensor("quantile prediction", output["quantile_preds"])
        loss = combined_loss(
            output["pred"], output["quantile_preds"],
            batch["y"], batch["mask"], self.cfg,
        )
        # sync_dist=True required for correct metric logging on multi-GPU
        self.log(
            f"{stage}_loss", loss,
            on_step=False, on_epoch=True, prog_bar=True,
            batch_size=batch["y"].size(0),
            sync_dist=False,   # single-GPU mode — sync_dist not needed
        )
        return loss

    def training_step(self,   batch, batch_idx): return self.shared_step(batch, "train")
    def validation_step(self, batch, batch_idx): return self.shared_step(batch, "val")
    def test_step(self,       batch, batch_idx): return self.shared_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.cfg.LR, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, self.cfg.MAX_EPOCHS), eta_min=self.cfg.LR_MIN)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

print("Forecaster defined.")


Forecaster defined.


In [16]:
# Standalone synthetic test: the corrected attention mask and model must be finite.

synthetic_model = ClusterTFT(
    n_past=len(cfg.DYNAMIC_PAST),
    n_future=len(cfg.DYNAMIC_FUTURE),
    n_static=54,
    hidden=cfg.TFT_HIDDEN,
    heads=cfg.TFT_HEADS,
    horizon=cfg.FORECAST_HORIZON,
    n_quantiles=len(cfg.QUANTILES),
    dropout=cfg.TFT_DROP,
    grn_hidden=cfg.TFT_GRN,
).to(DEVICE)
synthetic_model.eval()

with torch.no_grad():
    synthetic_output = synthetic_model(
        torch.randn(2, min(32, cfg.LOOKBACK), len(cfg.DYNAMIC_PAST), device=DEVICE),
        torch.randn(2, cfg.FORECAST_HORIZON, len(cfg.DYNAMIC_FUTURE), device=DEVICE),
        torch.randn(2, 54, device=DEVICE),
        return_attention=True,
    )

print("Synthetic predictions finite:", torch.isfinite(synthetic_output["pred"]).all().item())
print("Attention mask dtype:", synthetic_output["attn_mask"].dtype)
print("Attention mask contains NaN:", torch.isnan(synthetic_output["attn_mask"].float()).any().item())
assert torch.isfinite(synthetic_output["pred"]).all()
del synthetic_model, synthetic_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Synthetic predictions finite: True
Attention mask dtype: torch.bool
Attention mask contains NaN: False


In [17]:
# Optional station-quality file discovery

def find_quality_file():
    candidates = [
        cfg.DATA_ROOT / "Station_Quality_Classified.csv",
        cfg.DATA_ROOT.parent / "Station_Quality_Classified.csv",
    ]
    if Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").rglob("Station_Quality_Classified.csv"))
    for path in candidates:
        if Path(path).exists():
            return Path(path)
    return None

quality_path = find_quality_file()
quality_df = None
if quality_path is not None:
    raw_quality = pd.read_csv(quality_path)
    station_col = next((c for c in raw_quality.columns if str(c).strip().lower() in {"station", "gauge_id", "station_id"}), None)
    if station_col is not None and "Tier" in raw_quality.columns:
        quality_df = raw_quality.set_index(station_col)
        quality_df.index = quality_df.index.astype(str)
        print("Quality file:", quality_path)
        print(quality_df["Tier"].value_counts(dropna=False))
else:
    print("No station-quality file found; all listed basins will be used.")

No station-quality file found; all listed basins will be used.


In [18]:
# Training utilities

import time

def move_batch_to_device(batch, device):
    return {
        key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }


def preflight_forward(lit_model, dataloader):
    batch       = next(iter(dataloader))
    batch_dev   = move_batch_to_device(batch, DEVICE)
    lit_model   = lit_model.to(DEVICE).eval()
    with torch.no_grad():
        output = lit_model(batch_dev)
    diag = {
        "pred_finite"         : bool(torch.isfinite(output["pred"]).all().item()),
        "pred_nan"            : int(torch.isnan(output["pred"]).sum().item()),
        "x_past_nan"          : int(torch.isnan(batch_dev["x_past"]).sum().item()),
        "x_future_nan"        : int(torch.isnan(batch_dev["x_future"]).sum().item()),
        "x_static_nan"        : int(torch.isnan(batch_dev["x_static"]).sum().item()),
        "mask_true_fraction"  : float(batch_dev["mask"].float().mean().item()),
    }
    print("Preflight diagnostics:", diag)
    if not diag["pred_finite"]:
        raise FloatingPointError("Fresh model produced non-finite predictions.")
    lit_model.cpu().train()
    del batch_dev, output
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def load_lightning_state(lit_model, checkpoint_path):
    try:
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(checkpoint_path, map_location="cpu")
    lit_model.load_state_dict(ckpt["state_dict"])
    return lit_model


def plot_training_curve(log_csv_path, cluster_id, run_tag, output_dir):
    """
    Read Lightning metrics CSV and plot train vs val loss convergence.
    Called automatically after each cluster finishes training.
    """
    if not Path(log_csv_path).exists():
        print(f"  No metrics CSV found at {log_csv_path} — skipping convergence plot")
        return

    log = pd.read_csv(log_csv_path)

    # Lightning logs train_loss and val_loss in the same CSV with NaN rows
    train_col = "train_loss"
    val_col   = "val_loss"

    if train_col not in log.columns or val_col not in log.columns:
        print(f"  Columns not found. Available: {list(log.columns)}")
        return

    train_loss = log[log[train_col].notna()][train_col].values
    val_loss   = log[log[val_col].notna()][val_col].values
    epochs_t   = range(1, len(train_loss)+1)
    epochs_v   = range(1, len(val_loss)+1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: raw loss curves
    ax = axes[0]
    ax.plot(epochs_t, train_loss, 'b-',  lw=1.5, label='Train loss')
    ax.plot(epochs_v, val_loss,   'r--', lw=1.5, label='Val loss')
    best_epoch = int(np.argmin(val_loss)) + 1
    ax.axvline(best_epoch, color='green', ls=':', lw=1.2,
               label=f'Best epoch={best_epoch}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'Training curve — {run_tag} Cluster {cluster_id}')
    ax.legend(fontsize=8, frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Right: relative improvement per epoch (convergence indicator)
    ax = axes[1]
    if len(val_loss) > 1:
        rel_improvement = np.diff(val_loss) / (np.abs(val_loss[:-1]) + 1e-8)
        ax.plot(range(2, len(val_loss)+1), rel_improvement, 'g-', lw=1.5)
        ax.axhline(0, color='gray', ls='--', lw=0.8)
        ax.axhline(-0.001, color='orange', ls=':', lw=1,
                   label='Convergence threshold (0.1%)')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Relative val loss improvement')
        ax.set_title('Convergence indicator\n(negative = improvement)')
        ax.legend(fontsize=8, frameon=False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Print convergence summary
        n_improving = np.sum(rel_improvement < -0.001)
        print(f"  Convergence: {n_improving}/{len(rel_improvement)} epochs improved val loss > 0.1%")
        print(f"  Best val loss: {val_loss.min():.6f} at epoch {best_epoch}")
        if best_epoch == len(val_loss):
            print("  ⚠ Model was STILL IMPROVING at final epoch — consider more epochs or resume")
        else:
            print(f"  ✓ Model converged — best epoch {best_epoch}, "
                  f"early stopped at epoch {len(val_loss)}")

    fig.tight_layout()
    fig_path = Path(output_dir) / f"{run_tag}_C{cluster_id}_training_curve.png"
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Convergence plot saved: {fig_path}")
    return val_loss, best_epoch


def train_cluster(cluster_id, cfg, loader, quality_df=None, run_tag=None):
    run_tag = run_tag or cfg.RUN_TAG
    print("\n" + "=" * 72)
    print(f"Training {run_tag} — Cluster {cluster_id}")
    print("=" * 72)

    t_start = time.time()

    attributes = loader.load_attributes(cluster_id)
    tiers      = ["T1", "T2"] if quality_df is not None else None
    basin_data = loader.load_all_basins(cluster_id, tiers, quality_df)
    if len(basin_data) < 3:
        print(f"Only {len(basin_data)} usable basins — skipping.")
        return None

    attributes = attributes.reindex(list(basin_data.keys()))
    scaler     = HydroScaler().fit(basin_data, attributes, cfg)
    train_dl, val_dl, test_dl = make_dataloaders(basin_data, attributes, scaler, cfg)

    sample   = next(iter(train_dl))
    n_past   = sample["x_past"].shape[-1]
    n_future = sample["x_future"].shape[-1]
    n_static = sample["x_static"].shape[-1]
    print(f"Input dims: past={n_past}, future={n_future}, static={n_static}")
    print(f"Dataset:    train={len(train_dl.dataset):,} "
          f"val={len(val_dl.dataset):,} test={len(test_dl.dataset):,}")

    # Estimate epoch time
    n_batches = len(train_dl)
    print(f"Batches/epoch: {n_batches} | "
          f"Est. time @ 10 it/s: {n_batches/10/60:.0f} min/epoch | "
          f"Total est: {n_batches/10/60*cfg.MAX_EPOCHS/60:.1f} hrs")

    # Build model
    model = ClusterTFT(
        n_past=n_past, n_future=n_future, n_static=n_static,
        hidden=cfg.TFT_HIDDEN, heads=cfg.TFT_HEADS,
        horizon=cfg.FORECAST_HORIZON, n_quantiles=len(cfg.QUANTILES),
        dropout=cfg.TFT_DROP, grn_hidden=cfg.TFT_GRN,
    )
    lit_model = Forecaster(model, cfg, cluster_id)
    preflight_forward(lit_model, train_dl)

    # ── Resume from checkpoint if provided ────────────────────────────────────
    if cfg.RESUME_FROM_CHECKPOINT and Path(cfg.RESUME_FROM_CHECKPOINT).exists():
        lit_model = load_lightning_state(lit_model, cfg.RESUME_FROM_CHECKPOINT)
        print(f"  Resumed from: {cfg.RESUME_FROM_CHECKPOINT}")
    elif cfg.RESUME_FROM_CHECKPOINT:
        print(f"  WARNING: Checkpoint not found at {cfg.RESUME_FROM_CHECKPOINT}")
        print(f"  Starting from scratch.")

    # ── Callbacks ─────────────────────────────────────────────────────────────
    ckpt_dir = cfg.CKPT_DIR / f"{run_tag}_C{cluster_id}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    callbacks = [
        pl.callbacks.EarlyStopping(
            monitor="val_loss", patience=cfg.PATIENCE,
            mode="min", verbose=True,
            min_delta=1e-5          # stop if improvement < 0.00001
        ),
        pl.callbacks.ModelCheckpoint(
            dirpath=str(ckpt_dir),
            filename="best-{epoch:02d}-{val_loss:.4f}",
            monitor="val_loss", save_top_k=1, mode="min",
            save_last=True,         # also save last epoch for resume
        ),
        pl.callbacks.LearningRateMonitor(logging_interval="epoch"),
    ]

    # ── Trainer ───────────────────────────────────────────────────────────────
    # NOTE: ddp_notebook (multi-GPU fork) causes CUDA re-init error in Kaggle
    # Jupyter notebooks. Use single GPU (devices=1) which is stable and fast
    # enough for this dataset. T4 x2 means 16GB VRAM on device 0 — sufficient.
    trainer = pl.Trainer(
        max_epochs        = cfg.MAX_EPOCHS,
        accelerator       = "gpu" if torch.cuda.is_available() else "cpu",
        devices           = 1,          # single GPU — avoids CUDA fork error
        strategy          = "auto",     # no DDP in notebook environment
        precision         = cfg.PRECISION if torch.cuda.is_available() else "32-true",
        gradient_clip_val = cfg.GRAD_CLIP,
        callbacks         = callbacks,
        log_every_n_steps = 20,
        enable_progress_bar = True,
        deterministic     = False,
        default_root_dir  = str(cfg.OUTPUT_DIR),
    )

    print(f"\nUsing 1 GPU (single-device mode — stable in Kaggle notebooks)")
    trainer.fit(lit_model, train_dl, val_dl)

    # ── Timing ────────────────────────────────────────────────────────────────
    elapsed = (time.time() - t_start) / 60
    epochs_done = trainer.current_epoch + 1
    print(f"\nTraining complete: {epochs_done} epochs in {elapsed:.1f} min "
          f"({elapsed/epochs_done:.1f} min/epoch)")

    # ── Convergence plot ──────────────────────────────────────────────────────
    # Find the Lightning metrics CSV
    log_dir = Path(cfg.OUTPUT_DIR) / "lightning_logs"
    metrics_csvs = sorted(log_dir.rglob("metrics.csv")) if log_dir.exists() else []
    if metrics_csvs:
        plot_training_curve(metrics_csvs[-1], cluster_id, run_tag, cfg.OUTPUT_DIR)
    else:
        print("  No Lightning metrics CSV found — convergence plot skipped")

    # ── Load best checkpoint ──────────────────────────────────────────────────
    best_ckpts = sorted(ckpt_dir.glob("best-*.ckpt"))
    if best_ckpts:
        best_ckpt_path = str(best_ckpts[-1])
        lit_model = load_lightning_state(lit_model, best_ckpt_path)
        print(f"  Best checkpoint loaded: {Path(best_ckpt_path).name}")
    else:
        print("  WARNING: No best checkpoint found — using last epoch weights")

    # ── Save checkpoint explicitly to /kaggle/working for download ────────────
    save_dir = Path("/kaggle/working") / f"{run_tag}_C{cluster_id}_checkpoint"
    save_dir.mkdir(exist_ok=True)
    if best_ckpts:
        import shutil as _shutil
        _shutil.copy2(best_ckpts[-1], save_dir / "best.ckpt")
        print(f"  ✓ Checkpoint saved to {save_dir}/best.ckpt")
        print(f"    → Download this from Kaggle Output tab to resume in next session")

    return {
        "model"       : lit_model,
        "scaler"      : scaler,
        "test_loader" : test_dl,
        "ckpt_dir"    : ckpt_dir,
        "epochs_done" : epochs_done,
        "elapsed_min" : elapsed,
    }

print("train_cluster() defined with: resume | timing | convergence plot | multi-GPU")


train_cluster() defined with: resume | timing | convergence plot | multi-GPU


In [19]:
# Evaluation metrics and evaluator

def finite_pairs(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    valid = np.isfinite(obs) & np.isfinite(sim)
    return obs[valid], sim[valid]


def nse(obs, sim):
    obs, sim = finite_pairs(obs, sim)
    if len(obs) < 10:
        return np.nan
    denominator = np.sum((obs - obs.mean()) ** 2)
    if denominator <= 1e-12:
        return np.nan
    return 1.0 - np.sum((sim - obs) ** 2) / denominator


def kge(obs, sim):
    obs, sim = finite_pairs(obs, sim)
    if len(obs) < 10 or np.std(obs) <= 1e-12 or abs(np.mean(obs)) <= 1e-12:
        return np.nan
    if np.std(sim) <= 1e-12:
        correlation = 0.0
    else:
        correlation = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    return 1.0 - np.sqrt((correlation - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)


def weighted_nse(obs, sim, beta=2.0):
    obs, sim = finite_pairs(obs, sim)
    if len(obs) < 10:
        return np.nan
    weights = (np.maximum(obs, 0.0) + 1e-6) ** beta
    weights /= max(weights.mean(), 1e-12)
    weighted_mean = np.average(obs, weights=weights)
    denominator = np.sum(weights * (obs - weighted_mean) ** 2)
    if denominator <= 1e-12:
        return np.nan
    return 1.0 - np.sum(weights * (sim - obs) ** 2) / denominator


def flood_metrics(obs, sim, quantile=0.9):
    obs, sim = finite_pairs(obs, sim)
    if len(obs) < 10:
        return np.nan, np.nan, np.nan
    threshold = np.quantile(obs, quantile)
    observed_event = obs >= threshold
    simulated_event = sim >= threshold
    tp = np.sum(observed_event & simulated_event)
    fp = np.sum(~observed_event & simulated_event)
    fn = np.sum(observed_event & ~simulated_event)
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan
    pod = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    far = fp / (tp + fp) if (tp + fp) > 0 else np.nan
    return csi, pod, far


@torch.no_grad()
def evaluate_model(lit_model, test_loader, scaler, cfg, tag, cluster_id):
    lit_model = lit_model.to(DEVICE).eval()
    collected = defaultdict(lambda: defaultdict(lambda: {"obs": [], "pred": []}))

    for batch in tqdm(test_loader, desc=f"Evaluating C{cluster_id}"):
        device_batch = move_batch_to_device(batch, DEVICE)
        output = lit_model(device_batch)
        if not torch.isfinite(output["pred"]).all():
            raise FloatingPointError(f"Non-finite predictions during evaluation of cluster {cluster_id}")

        pred_scaled = output["pred"].detach().cpu().numpy()
        obs_scaled = batch["y"].numpy()
        mask = batch["mask"].numpy().astype(bool)

        for i, bid in enumerate(batch["basin_id"]):
            pred = scaler.inverse_q(bid, pred_scaled[i])
            obs = scaler.inverse_q(bid, obs_scaled[i])
            for lead in cfg.LEAD_TIMES:
                idx = lead - 1
                if mask[i, idx]:
                    collected[bid][lead]["obs"].append(float(obs[idx]))
                    collected[bid][lead]["pred"].append(float(pred[idx]))

    rows = []
    for bid, lead_data in collected.items():
        for lead in cfg.LEAD_TIMES:
            obs = np.asarray(lead_data[lead]["obs"], dtype=float)
            pred = np.asarray(lead_data[lead]["pred"], dtype=float)
            csi, pod, far = flood_metrics(obs, pred)
            rmse = float(np.sqrt(np.mean((pred - obs) ** 2))) if len(obs) else np.nan
            pbias = float(100.0 * np.sum(pred - obs) / np.sum(obs)) if len(obs) and np.sum(obs) > 1e-12 else np.nan
            rows.append({
                "model": tag,
                "cluster": cluster_id,
                "basin_id": bid,
                "lead_time": lead,
                "NSE": nse(obs, pred),
                "KGE": kge(obs, pred),
                "wNSE": weighted_nse(obs, pred),
                "RMSE": rmse,
                "PBIAS": pbias,
                "CSI": csi,
                "POD": pod,
                "FAR": far,
                "n_obs": int(len(obs)),
            })

    result = pd.DataFrame(rows)
    if result.empty:
        raise RuntimeError(f"Evaluation produced no rows for cluster {cluster_id}")
    return result

print("Evaluation functions defined.")

Evaluation functions defined.


In [ ]:
# Train and evaluate all selected clusters

all_results      = []
cluster_artifacts = {}

for cluster_id in cfg.CLUSTERS:
    artifact = train_cluster(cluster_id, cfg, loader, quality_df,
                             run_tag=cfg.RUN_TAG)
    if artifact is None:
        continue

    results = evaluate_model(
        artifact["model"], artifact["test_loader"], artifact["scaler"],
        cfg, cfg.RUN_TAG, cluster_id,
    )
    result_path = cfg.OUTPUT_DIR / f"{cfg.RUN_TAG}_C{cluster_id}_results.csv"
    results.to_csv(result_path, index=False)
    all_results.append(results)
    cluster_artifacts[cluster_id] = artifact

    print(f"\nCluster {cluster_id} — elapsed: {artifact['elapsed_min']:.0f} min | "
          f"epochs: {artifact['epochs_done']}")
    print(results.groupby("lead_time")[["NSE","KGE","wNSE","CSI"]].median().round(3))

    # Free GPU memory before next cluster
    cluster_artifacts[cluster_id]["model"] = artifact["model"].cpu()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

if not all_results:
    raise RuntimeError("No cluster completed successfully.")

combined      = pd.concat(all_results, ignore_index=True)
combined_path = cfg.OUTPUT_DIR / f"{cfg.RUN_TAG}_all_results.csv"
combined.to_csv(combined_path, index=False)

summary      = combined.groupby(["cluster","lead_time"])[["NSE","KGE","wNSE","CSI"]].median()
summary_path = cfg.OUTPUT_DIR / f"{cfg.RUN_TAG}_median_summary.csv"
summary.to_csv(summary_path)

print("\nFinal median summary:")
print(summary.round(3))
print("\nSaved:", combined_path)
print("Saved:", summary_path)



Training TFT_global — Cluster 0


Loading Cluster 0:   0%|          | 0/206 [00:00<?, ?it/s]

Cluster 0: loaded 206/206 basins
Samples: train=1,498,856, val=520,768, test=445,372
Input dims: past=5, future=4, static=54
Dataset:    train=1,498,856 val=520,768 test=445,372
Batches/epoch: 11710 | Est. time @ 10 it/s: 20 min/epoch | Total est: 19.5 hrs


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Preflight diagnostics: {'pred_finite': True, 'pred_nan': 0, 'x_past_nan': 0, 'x_future_nan': 0, 'x_static_nan': 0, 'mask_true_fraction': 0.9510416984558105}

Using 1 GPU (single-device mode — stable in Kaggle notebooks)


2026-07-13 07:00:17.928477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783926018.114270      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783926018.169596      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783926018.617332      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783926018.617365      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783926018.617368      58 computation_placer.cc:177] computation placer alr

┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ClusterTFT │  1.0 M │ train │     0 │
└───┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.0 M                                                                                                
Total estimated model params size (MB): 4.178                                                                      
Modules in train mode: 170                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 73.687
Metric val_loss improved by 50.061 >= min_delta = 1e-05. New best score: 23.626
Metric val_loss improved by 1.379 >= min_delta = 1e-05. New best score: 22.247
Metric val_loss improved by 7.271 >= min_delta = 1e-05. New best score: 14.976
Metric val_loss improved by 4.625 >= min_delta = 1e-05. New best score: 10.351
Metric val_loss improved by 0.718 >= min_delta = 1e-05. New best score: 9.633
Metric val_loss improved by 0.848 >= min_delta = 1e-05. New best score: 8.785
Metric val_loss improved by 1.160 >= min_delta = 1e-05. New best score: 7.625


In [ ]:
# ── Forecast skill plot ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if all_results:
    combined = pd.concat(all_results, ignore_index=True)

    LEADS      = cfg.LEAD_TIMES
    CL_COLORS  = {0:'#888780', 1:'#E24B4A', 2:'#378ADD', 3:'#1D9E75', 4:'#EF9F27'}
    CL_NAMES   = {0:'Global (all basins)', 1:'C1 Central Inland',
                  2:'C2 Eastern Plateau', 3:'C3 Southern Peninsula',
                  4:'C4 North-Central'}

    metrics = [
        ("NSE",  "Nash-Sutcliffe Efficiency",      "A"),
        ("KGE",  "Kling-Gupta Efficiency",          "B"),
        ("wNSE", "Flow-weighted NSE (flood peaks)", "C"),
        ("CSI",  "Critical Success Index (Q90)",    "D"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for ax, (met, ylabel, panel) in zip(axes.flat, metrics):
        for cl in sorted(combined["cluster"].unique()):
            sub  = combined[combined["cluster"] == cl]
            med  = sub.groupby("lead_time")[met].median()
            q25  = sub.groupby("lead_time")[met].quantile(0.25)
            q75  = sub.groupby("lead_time")[met].quantile(0.75)
            col  = CL_COLORS.get(int(cl), "#888780")
            name = CL_NAMES.get(int(cl), f"C{cl}")
            ls   = "--" if cl == 0 else "-"
            ax.plot(LEADS, [med.get(lt, np.nan) for lt in LEADS],
                    marker="o", ls=ls, lw=2, color=col, label=name)
            ax.fill_between(LEADS,
                            [q25.get(lt, np.nan) for lt in LEADS],
                            [q75.get(lt, np.nan) for lt in LEADS],
                            alpha=0.10, color=col)

        ax.axhline(0,   color="gray", ls="--", lw=0.8, alpha=0.5)
        if met in ("NSE", "KGE", "wNSE"):
            ax.axhline(0.5, color="gray", ls=":", lw=0.8, alpha=0.5)
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel(met)
        ax.set_title(f"{panel}   {ylabel}")
        ax.set_xticks(LEADS)
        ax.legend(fontsize=7.5, frameon=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        f"TFT forecast skill — {cfg.RUN_TAG}\n"
        f"(line=cluster median; shading=IQR; dashed=Global baseline)",
        fontsize=11, fontweight="bold"
    )
    fig.tight_layout()
    fig_path = cfg.OUTPUT_DIR / f"{cfg.RUN_TAG}_skill_comparison.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

    # ── Print timing summary ──────────────────────────────────────────────────
    print("\n=== TRAINING SUMMARY ===")
    total_min = sum(a["elapsed_min"] for a in cluster_artifacts.values()
                    if "elapsed_min" in a)
    for cl, a in cluster_artifacts.items():
        if "elapsed_min" in a:
            print(f"  Cluster {cl}: {a['epochs_done']} epochs in "
                  f"{a['elapsed_min']:.0f} min "
                  f"({a['elapsed_min']/a['epochs_done']:.1f} min/epoch)")
    print(f"  Total training time: {total_min:.0f} min ({total_min/60:.1f} hrs)")


In [ ]:
# Final integrity check and archive for download from Kaggle's Output panel

result_files = sorted(cfg.OUTPUT_DIR.rglob("*"))
print(f"Generated {len([p for p in result_files if p.is_file()])} files.")
for path in result_files:
    if path.is_file():
        print(path.relative_to(cfg.OUTPUT_DIR))

archive_base = str(WORK_ROOT / "tft_corrected_outputs")
archive_path = shutil.make_archive(archive_base, "zip", root_dir=cfg.OUTPUT_DIR)
print("Archive created:", archive_path)